# CSE 153/253 Assignment 2 Workbook

This workbook supports the final presentation for symbolic music generation. The project trains all weights from scratch on the Nottingham Dataset and produces two symbolic MIDI files: unconditioned melody generation and chord-conditioned melody generation.

## Dataset Investigation

Candidate datasets considered:

| Dataset | Format | Strengths | Weaknesses | Decision |
|---|---|---|---|---|
| Nottingham | ABC symbolic folk tunes with melodies and chord symbols | Small, clean, fast to train, directly supports chord-conditioned melody generation | Less stylistic variety than larger corpora | Selected |
| MAESTRO | Aligned MIDI/audio piano performances | High-quality expressive piano data | Large, piano-only, no explicit chord conditioning | Not selected |
| Lakh MIDI | Large MIDI collection | Huge symbolic corpus | Noisy, heavy preprocessing burden | Not selected |
| JSB Chorales | Symbolic chorales | Clean polyphonic harmony | Small and stylistically narrow | Not selected |
| Essen Folk | ABC folk melodies | Larger folk corpus | More heterogeneous than Nottingham | Backup option |

Nottingham is the best fit because it gives us melody and chord information in a manageable symbolic format. That keeps the project realistic while still supporting a strong Task 2 conditioning story.

## Modeling Investigation

| Model | Role | Difficulty | Cost | Expected Quality | Why |
|---|---|---|---|---|---|
| Markov chain | Baseline | Low | Very low | Low-medium | Easy comparison point for local note transitions |
| N-gram | Baseline | Low | Very low | Medium | Better symbolic baseline using short context |
| LSTM | Neural candidate | Medium | Moderate | Good | Strong sequence model but slightly more parameters than GRU |
| GRU | Main model | Medium | Moderate-low | Good | Strong enough for the assignment, easy to explain, fast to train |
| Transformer | Stretch option | Higher | Higher | Potentially high | More complex than needed for this assignment timeline |

The final architecture is a GRU next-token model. It balances assignment quality, development speed, and presentation clarity.

## Data Pipeline

1. Download Nottingham ABC files.
2. Parse symbolic scores with `music21`.
3. Extract the melody part, note pitches, rests, durations, bar boundaries, and chord symbols.
4. Quantize durations into sixteenth-note units.
5. Build two token streams:
   - Task 1: `<START> NOTE_pitch_duration ... <BAR> ... <END>`
   - Task 2: `<START> CHORD_C NOTE_pitch_duration ... <BAR> CHORD_G ... <END>`
6. Split by tune into train and validation sets.
7. Train next-token prediction models from scratch.

In [ ]:
import json
from pathlib import Path

summary_path = Path('../data/processed/nottingham/summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    summary
else:
    print('Run preprocessing first to populate dataset summary.')

## Task 1: Unconditioned Melody Generation

- Input during training: previous melody tokens.
- Output during training: next melody token.
- Objective: cross-entropy next-token prediction.
- Generation: start with `<START>`, sample autoregressively, stop at `<END>` or maximum length.
- Deliverable: `symbolic_unconditioned.mid`.

## Task 2: Chord-Conditioned Melody Generation

- Input during training: chord tokens interleaved with melody tokens.
- Conditioning mechanism: the chord token appears before each measure, so the GRU hidden state carries the chord context while predicting notes.
- Objective: cross-entropy next-token prediction.
- Generation: feed a chord progression, sample melody tokens until a bar token, then feed the next chord.
- Deliverable: `symbolic_conditioned.mid`.

## Evaluation Design

Objective metrics:

- Validation loss and perplexity: measures predictive learning on held-out tunes.
- Pitch diversity: checks whether the model avoids collapsing to a few notes.
- Duration diversity: checks rhythmic variety.
- Repetition rate: detects loops and overly repetitive samples.
- Pitch range: confirms generated melodies stay in a plausible musical register.

Subjective evaluation:

- Listen to both generated MIDI files.
- Compare against n-gram baseline samples.
- Rate coherence, style similarity, repetition, and chord fit.

In [ ]:
for path in [
    Path('../outputs/checkpoints/unconditioned/metrics.json'),
    Path('../outputs/checkpoints/conditioned/metrics.json'),
]:
    if path.exists():
        print(path)
        print(json.dumps(json.loads(path.read_text())[-3:], indent=2))
    else:
        print(f'Missing {path}; train the model first.')

## Related Work Discussion

This project is framed as symbolic language modeling: musical events are discrete tokens, and generation is next-token sampling. This connects naturally to Markov models, n-gram models, recurrent neural networks, and modern transformer music models. The GRU is intentionally chosen as a practical middle ground: it demonstrates learned sequence modeling without requiring pretrained weights or a large compute budget.